# Ford Mustang Mach-E 2026 — ERG + Rescue Sheet → Pinecone
### With proper image handling via `orig_elements` + GPT-4o summarization

**Key fix over v1:** `Image` elements embedded in chunks are no longer silently dropped.
After partitioning, we walk each chunk's `orig_elements`, decode any `image_base64`
fields, summarize them with GPT-4o vision, and append the summary to the chunk text
before upserting to Pinecone.

This matters a lot for these PDFs — roughly half the content is diagrams:
- HV component location maps (p.3, rescue sheet p.1)
- No-cut zone diagram (p.21)
- Stabilization/lifting point diagram (p.9)
- Fire assessment flame/smoke diagrams (p.30–32)
- Airbag location top-view (p.26)


## 1. Install Dependencies

In [1]:
# System dependencies (run once in terminal if not installed):
#   macOS: brew install poppler tesseract
#   Linux: sudo apt-get install poppler-utils tesseract-ocr

!pip install -q --upgrade pillow
!pip install -q openai
!pip install -q pandas
!pip install -q --upgrade nltk
!pip install -q python-dotenv
!pip install -q langchain langchain-openai langchain-community langchain-pinecone
!pip install -q unstructured unstructured-inference
!pip install -q pdfminer.six pdf2image pi-heif pytesseract unstructured-pytesseract
!pip install -q pinecone-client


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


## 2. Imports & Setup

In [2]:
import os
import base64
import textwrap
from collections import Counter

from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY, PINECONE_API_KEY from .env

# Unstructured — local (no API key needed)
from unstructured.partition.pdf import partition_pdf as _partition_pdf_local
from unstructured.chunking.title import chunk_by_title
from unstructured.staging.base import (
    elements_from_base64_gzipped_json,
)

# OpenAI (for image summarization)
from openai import OpenAI

# LangChain / Pinecone
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

print("Imports loaded.")

Imports loaded.


## 3. Configuration

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_PATH = os.path.join(os.getcwd(), "Ford Mache-E")

DOCS = [
    {
        "filename": "EmergencyResponseGuide-Ford-Mach-E-2026.pdf",
        "namespace": "erg_full",
        "doc_type":  "emergency_response_guide",
    },
    {
        "filename": "RescueSheet-Ford-Mach-E-2026.pdf",
        "namespace": "rescue_sheet",
        "doc_type":  "rescue_sheet",
    },
]

PROCESSED_LOG = os.path.join(DRIVE_PATH, "processed.log")

# ── Pinecone ───────────────────────────────────────────────────────────────
PINECONE_INDEX_NAME = "ford-mache-erg"
PINECONE_CLOUD      = "aws"
PINECONE_REGION     = "us-east-1"

# ── Unstructured chunking ──────────────────────────────────────────────────
CHUNKING_STRATEGY  = "by_title"
MAX_CHARACTERS     = 2000
COMBINE_UNDER      = 500
NEW_AFTER_N        = 1800

# ── Image summarization ────────────────────────────────────────────────────
IMAGE_SYSTEM_PROMPT = """You are an expert in vehicle emergency response.
You are helping index a Ford Mustang Mach-E 2026 Emergency Response Guide
for first responders.

When shown a diagram or image from this document, write a concise but
complete plain-text description that captures:
1. What the image shows (e.g. "top-down diagram of the vehicle underside")
2. All labeled components, zones, or callouts visible
3. Any safety-critical information (no-cut zones, HV battery location,
   recommended lift points, etc.)

Write the description in 3-8 sentences. Do not use markdown.
A first responder querying a RAG system will read this description."""

# ── API Keys (loaded from .env) ───────────────────────────────────────────
# .env should contain: OPENAI_API_KEY and PINECONE_API_KEY
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print(f"Source files: {DRIVE_PATH}")
print("Configuration loaded.")

Source files: /Users/18668005@nebraska.edu/Documents/RAG_Responder/Ford Mache-E
Configuration loaded.


## 4. Image Summarization Helpers

These functions implement the `orig_elements` recovery pattern from the
[Unstructured chunking docs](https://docs.unstructured.io/open-source/core-functionality/chunking#recovering-chunk-elements).

Key doc quote:
> *If High Res partitioning is used, the `orig_elements` value of each chunked
> element will contain an `image_base64` value for each of the `Image` and
> `Table` elements.*

In [4]:
def decode_orig_elements(chunk):
    """
    Recover the original sub-elements that were merged into this chunk.

    Unstructured serialises orig_elements as Base64-gzipped JSON.
    We use elements_from_base64_gzipped_json() to decode it.
    Returns [] if the field is absent or fails to decode.

    Ref: https://docs.unstructured.io/open-source/core-functionality/chunking
         #recovering-chunk-elements
    """
    try:
        meta_dict = chunk.metadata.to_dict()
        raw = meta_dict.get("orig_elements")
        if not raw:
            return []
        return elements_from_base64_gzipped_json(raw)
    except Exception as e:
        print(f"    ⚠️  Could not decode orig_elements: {e}")
        return []


def summarize_image_b64(image_b64: str, context_text: str = "") -> str:
    """
    Send a base64-encoded image to GPT-4o vision and return a plain-text
    description suited for RAG retrieval.

    `context_text` is the surrounding chunk text — we pass it as a user hint
    so GPT-4o can label callouts correctly (e.g. "the orange cables shown
    correspond to high-voltage wiring mentioned in the surrounding text").
    """
    user_content = [
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{image_b64}",
                "detail": "high",   # use high detail — these diagrams are dense
            },
        }
    ]

    if context_text.strip():
        user_content.append({
            "type": "text",
            "text": (
                f"The surrounding text in this section reads:\n"
                f"{context_text[:600]}\n\n"
                "Describe the diagram above in the context of this section."
            ),
        })
    else:
        user_content.append({
            "type": "text",
            "text": "Describe this diagram for a first responder RAG system.",
        })

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": IMAGE_SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ],
        max_tokens=400,
        temperature=0.2,   # low temp — factual descriptions only
    )
    return response.choices[0].message.content.strip()


def enrich_chunk_with_image_summaries(chunk) -> tuple[str, int]:
    """
    Given a single Unstructured chunk element, find any Image sub-elements
    inside orig_elements, summarize them, and return:
      - enriched_text : original chunk text + appended image summaries
      - n_images      : how many images were found and summarized

    Pattern from Unstructured docs:
        [e.metadata.image_path
         for e in chunk.metadata.orig_elements
         if e.metadata.image_path is not None]
    We use image_base64 instead of image_path because we're working with
    the API (no local filesystem).
    """
    base_text  = str(chunk.text).strip()
    summaries  = []
    orig_els   = decode_orig_elements(chunk)

    for orig_el in orig_els:
        el_type = type(orig_el).__name__

        # Only process Image elements (Tables are handled as structured text)
        if el_type != "Image":
            continue

        orig_meta = orig_el.metadata.to_dict() if hasattr(orig_el.metadata, "to_dict") else {}
        image_b64 = orig_meta.get("image_base64")

        if not image_b64:
            # Fallback: try image_path (only present in local/open-source mode)
            image_path = orig_meta.get("image_path")
            if image_path and os.path.exists(image_path):
                with open(image_path, "rb") as img_f:
                    image_b64 = base64.b64encode(img_f.read()).decode("utf-8")

        if not image_b64:
            continue

        print(f"      🖼️  Summarizing image ({el_type}) on p."
              f"{orig_meta.get('page_number', '?')}...")

        summary = summarize_image_b64(image_b64, context_text=base_text)
        summaries.append(f"[Image description: {summary}]")

    enriched = base_text
    if summaries:
        enriched = base_text + "\n\n" + "\n\n".join(summaries)

    return enriched, len(summaries)


print("Image summarization helpers defined.")

Image summarization helpers defined.


## 5. Core Pipeline Helpers

In [5]:
def get_processed_files():
    if not os.path.exists(PROCESSED_LOG):
        return set()
    with open(PROCESSED_LOG, 'r') as f:
        return set(f.read().splitlines())

def mark_file_processed(filename):
    with open(PROCESSED_LOG, 'a') as f:
        f.write(filename + '\n')


def setup_pinecone():
    pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
    existing = [idx.name for idx in pc.list_indexes()]
    if PINECONE_INDEX_NAME not in existing:
        print(f"Creating Pinecone index '{PINECONE_INDEX_NAME}'...")
        pc.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=1536,
            metric='cosine',
            spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
    else:
        print(f'Using existing index \'{PINECONE_INDEX_NAME}\'')
    return pc


def partition_pdf(pdf_path, pdf_filename):
    """
    Partition a PDF locally via unstructured, then chunk by_title.
    No API key required — runs entirely on-machine.
    """
    print(f"  Partitioning '{pdf_filename}' locally...")

    elements = _partition_pdf_local(
        filename=pdf_path,
        strategy='auto',          # 'auto' picks hi_res+OCR for complex/image-heavy PDFs
        infer_table_structure=True,
    )

    chunks = chunk_by_title(
        elements,
        max_characters=MAX_CHARACTERS,
        combine_text_under_n_chars=COMBINE_UNDER,
        new_after_n_chars=NEW_AFTER_N,
        multipage_sections=True,
    )

    print(f"  Got {len(chunks)} chunks.")
    return chunks


def elements_to_langchain_docs(elements, filename, namespace, doc_type):
    """
    Convert Unstructured chunk Elements -> LangChain Documents.
    Enriches each chunk with GPT-4o image summaries where available.
    """
    docs       = []
    total_imgs = 0

    for i, chunk in enumerate(elements):
        raw_text = str(chunk.text).strip()
        if not raw_text:
            continue

        meta = chunk.metadata.to_dict() if hasattr(chunk.metadata, 'to_dict') else {}

        enriched_text, n_imgs = enrich_chunk_with_image_summaries(chunk)
        total_imgs += n_imgs

        if n_imgs:
            print(f'    chunk {i:03d}: added {n_imgs} image summary/ies')

        orig_els  = decode_orig_elements(chunk)
        all_pages = sorted({
            e.metadata.to_dict().get('page_number')
            for e in orig_els
            if e.metadata.to_dict().get('page_number') is not None
        })
        page_number  = meta.get('page_number') or (all_pages[0] if all_pages else None)
        page_span    = f"{all_pages[0]}-{all_pages[-1]}" if len(all_pages) > 1 else str(page_number)
        section_title = meta.get('section') or meta.get('parent_id') or 'unknown'

        lc_doc = Document(
            page_content=enriched_text,
            metadata={
                'source_doc':    filename,
                'doc_type':      doc_type,
                'namespace':     namespace,
                'vehicle':       'Ford Mustang Mach-E 2026',
                'page_number':   page_number,
                'page_span':     page_span,
                'section_title': str(section_title),
                'element_type':  type(chunk).__name__,
                'chunk_index':   i,
                'has_images':    n_imgs > 0,
                'image_count':   n_imgs,
            },
        )
        docs.append(lc_doc)

    print(f'  -> Built {len(docs)} LangChain docs | {total_imgs} images summarized')
    return docs


print("Pipeline helpers defined.")

Pipeline helpers defined.


## 6. (Optional) Single Chunk Inspection

Run this on a small number of elements to verify `orig_elements` decoding
and image detection before running the full loop.

In [6]:
def inspect_chunk_images(elements, max_chunks=10):
    """
    Walk the first N chunks and report what orig_elements each contains.
    Useful for verifying that image_base64 is present before doing full run.
    """
    print(f"\n--- Inspecting orig_elements for first {max_chunks} chunks ---")
    for i, chunk in enumerate(elements[:max_chunks]):
        orig_els = decode_orig_elements(chunk)
        type_counts = Counter(type(e).__name__ for e in orig_els)
        has_img_b64 = any(
            e.metadata.to_dict().get("image_base64")
            for e in orig_els
            if type(e).__name__ == "Image"
        )
        print(
            f"  chunk {i:02d} | sub-elements: {dict(type_counts)} "
            f"| image_base64 present: {has_img_b64}"
        )


# Test against the rescue sheet (4 pages, faster)
TEST_FILE = "RescueSheet-Ford-Mach-E-2026.pdf"
test_path = os.path.join(DRIVE_PATH, TEST_FILE)

test_elements = partition_pdf(test_path, TEST_FILE)
inspect_chunk_images(test_elements, max_chunks=8)

# Quick look at element type mix
print("\n--- Element type counts ---")
print(Counter(type(e).__name__ for e in test_elements))

No languages specified, defaulting to English.


  Partitioning 'RescueSheet-Ford-Mach-E-2026.pdf' locally...


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

  Got 4 chunks.

--- Inspecting orig_elements for first 8 chunks ---
  chunk 00 | sub-elements: {'Image': 5, 'Text': 9, 'Title': 1, 'NarrativeText': 2} | image_base64 present: False
  chunk 01 | sub-elements: {'Title': 1, 'NarrativeText': 3, 'Image': 4, 'Text': 4} | image_base64 present: False
  chunk 02 | sub-elements: {'Title': 2, 'NarrativeText': 2, 'Text': 2, 'Image': 1} | image_base64 present: False
  chunk 03 | sub-elements: {'Title': 1, 'NarrativeText': 2, 'ListItem': 1, 'Image': 1, 'Text': 3} | image_base64 present: False

--- Element type counts ---
Counter({'CompositeElement': 4})


## 7. Full Processing Loop → Pinecone

In [7]:
pc            = setup_pinecone()
embeddings    = OpenAIEmbeddings(model="text-embedding-3-small")
processed     = get_processed_files()

print(f"Already processed: {processed or 'none'}")

for doc_cfg in DOCS:
    filename  = doc_cfg["filename"]
    namespace = doc_cfg["namespace"]
    doc_type  = doc_cfg["doc_type"]
    pdf_path  = os.path.join(DRIVE_PATH, filename)

    print(f"\n{'─'*60}")
    print(f"📄  {filename}  →  namespace: {namespace}")

    if filename in processed:
        print("    ⏭️  Already processed — skipping.")
        continue

    if not os.path.exists(pdf_path):
        print(f"    ❌ Not found at {pdf_path}")
        continue

    try:
        # Step A: Parse + chunk via Unstructured API
        elements = partition_pdf(pdf_path, filename)
        if not elements:
            print("    ⚠️  No elements — skipping.")
            continue

        print(f"  Element type breakdown:")
        for etype, cnt in Counter(type(e).__name__ for e in elements).items():
            print(f"    {etype}: {cnt}")

        # Step B: Convert → LangChain docs, with image summarization
        print("  Enriching chunks with image summaries...")
        lc_docs = elements_to_langchain_docs(elements, filename, namespace, doc_type)

        # Step C: Upsert to Pinecone
        print(f"  Upserting {len(lc_docs)} docs to '{PINECONE_INDEX_NAME}/{namespace}'...")
        PineconeVectorStore.from_documents(
            documents=lc_docs,
            embedding=embeddings,
            index_name=PINECONE_INDEX_NAME,
            namespace=namespace,
        )

        mark_file_processed(filename)
        print(f"  ✅ Done — '{filename}' indexed.")

    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        raise   # re-raise so you see the full traceback

print(f"\n{'─'*60}\nAll documents processed.")

Using existing index 'ford-mache-erg'
Already processed: none

────────────────────────────────────────────────────────────
📄  EmergencyResponseGuide-Ford-Mach-E-2026.pdf  →  namespace: erg_full
  Partitioning 'EmergencyResponseGuide-Ford-Mach-E-2026.pdf' locally...


No languages specified, defaulting to English.


  Got 37 chunks.
  Element type breakdown:
    CompositeElement: 36
    Table: 1
  Enriching chunks with image summaries...
  -> Built 37 LangChain docs | 0 images summarized
  Upserting 37 docs to 'ford-mache-erg/erg_full'...


No languages specified, defaulting to English.


  ✅ Done — 'EmergencyResponseGuide-Ford-Mach-E-2026.pdf' indexed.

────────────────────────────────────────────────────────────
📄  RescueSheet-Ford-Mach-E-2026.pdf  →  namespace: rescue_sheet
  Partitioning 'RescueSheet-Ford-Mach-E-2026.pdf' locally...


  Got 4 chunks.
  Element type breakdown:
    CompositeElement: 4
  Enriching chunks with image summaries...
  -> Built 4 LangChain docs | 0 images summarized
  Upserting 4 docs to 'ford-mache-erg/rescue_sheet'...


  ✅ Done — 'RescueSheet-Ford-Mach-E-2026.pdf' indexed.

────────────────────────────────────────────────────────────
All documents processed.


## 8. Smoke Tests

In [8]:
def query(namespace, q, k=3):
    store   = PineconeVectorStore(
        index_name=PINECONE_INDEX_NAME,
        embedding=embeddings,
        namespace=namespace,
    )
    results = store.similarity_search(q, k=k)
    print(f"\n🔍  [{namespace}]  '{q}'")
    for i, doc in enumerate(results):
        meta = doc.metadata
        print(f"  [{i+1}] p.{meta.get('page_span')}  "
              f"type={meta.get('element_type')}  "
              f"imgs={meta.get('image_count',0)}")
        print(textwrap.fill(doc.page_content[:350], 76,
                            initial_indent="      ",
                            subsequent_indent="      "))

# Text-heavy queries
query("erg_full",    "how to disable the high voltage system urgent")
query("erg_full",    "HV battery fire thermal runaway water cooling")
query("erg_full",    "submerged in water fizzing bubbling battery")

# Image-dependent queries — these will only return useful results
# if image summarization ran correctly
query("erg_full",    "no cut zones diagram side of vehicle")
query("erg_full",    "orange high voltage cables location underside")
query("rescue_sheet","airbag locations top view")


🔍  [erg_full]  'how to disable the high voltage system urgent'
  [1] p.13-15  type=CompositeElement  imgs=0.0
      OPTION 2 - Under non-urgent situations  1. Ensure the vehicle
      transmission gear selector is in the PARK position. Check that the
      vehicle READY light is off to verify the high voltage system is
      disconnected. If the vehicle READY light is on, press the engine
      Start/Stop button to turn off the ignition.  —_  CPR © 2025 FORD MOTOR
      COMPANY DEARBORN, MIC
  [2] p.18-19  type=CompositeElement  imgs=0.0
      Service and handling of Pyrotechnic Components is restricted to
      qualified personnel. The required qualifications vary by region.
      Always observe local laws and legislative directives regarding
      Pyrotechnic Components service and handling. Failure to follow this
      instruction may result in serious personal injury or death.  If
      occupant removal is necess
  [3] p.34-35  type=CompositeElement  imgs=0.0
      Follow the guid


🔍  [erg_full]  'HV battery fire thermal runaway water cooling'
  [1] p.29-30  type=CompositeElement  imgs=0.0
      DO NOT ATTEMPT TO DISCHARGE THE HV BATTERY.  White/light gray smoke
      from a HV battery contains chemicals which can be:  ¢ Combustible
      where the smoke is exiting the battery vent location  ¢ Explosive in a
      confined space environment (such as a small garage) with the correct
      mixture of air  ¢ Toxic in short exposure durations  Forcing water
      into a HV
  [2] p.32  type=CompositeElement  imgs=0.0
      1. Engage the HV Battery to extinguish any flames and cool all
      accessible surfaces  a. Apply water onto the HV Battery  i. Direct
      application of water is best  ii. Tilting the vehicle may improve HV
      Battery access to apply water. Follow the precautions in Section 2 for
      Stabilization and Lifting Points.  b. Continue to apply water for
      several minute
  [3] p.31  type=CompositeElement  imgs=0.0
      2. Contain any active


🔍  [erg_full]  'submerged in water fizzing bubbling battery'
  [1] p.33-34  type=CompositeElement  imgs=0.0
      FORD Mustang Mach-E 2026  7. In case of submersion  A WARNING:
      DAMAGED ELECTRIC VEHICLES SUBMERGED IN WATER PRESENT A POTENTIAL HIGH
      VOLTAGE ELECTRICAL SHOCK HAZARD. EXERCISE CAUTION AND WEAR APPROPRIATE
      PERSONAL PROTECTIVE EQUIPMENT (PPE) INCLUDING HIGH VOLTAGE SAFETY
      GLOVES AND BOOTS. REMOVE ALL METALLIC JEWELRY, INCLUDING WATCHES AND
      RINGS. DO
  [2] p.32  type=CompositeElement  imgs=0.0
      1. Engage the HV Battery to extinguish any flames and cool all
      accessible surfaces  a. Apply water onto the HV Battery  i. Direct
      application of water is best  ii. Tilting the vehicle may improve HV
      Battery access to apply water. Follow the precautions in Section 2 for
      Stabilization and Lifting Points.  b. Continue to apply water for
      several minute
  [3] p.29-30  type=CompositeElement  imgs=0.0
      DO NOT ATTEMPT TO DISC


🔍  [erg_full]  'no cut zones diagram side of vehicle'
  [1] p.18-19  type=CompositeElement  imgs=0.0
      Service and handling of Pyrotechnic Components is restricted to
      qualified personnel. The required qualifications vary by region.
      Always observe local laws and legislative directives regarding
      Pyrotechnic Components service and handling. Failure to follow this
      instruction may result in serious personal injury or death.  If
      occupant removal is necess
  [2] p.6-8  type=CompositeElement  imgs=0.0
      FORD Mustang Mach-E  2026  Instrument Cluster  A Cc | ve @ee (30 Gee
      XX% eee" Qo pax *8.*.*. eee. w. J 35  A - High voltage battery gauge
      B - Driver assist message display area  C - Driver assist area  D -
      Speedometer  E - Pop-up message display area  F - Gear indicator  G -
      Odometer  H - Compass  | - Information Bar  J - Range display area  K
      - P
  [3] p.21-22  type=CompositeElement  imgs=0.0
      FORD Mustang Mach-E 2026  N


🔍  [erg_full]  'orange high voltage cables location underside'
  [1] p.9-10  type=CompositeElement  imgs=0.0
      FORD Mustang Mach-E 2026  STABILIZATION / LIFTING POINTS  The high
      voltage battery is located behind an underbody air shield under the
      vehicle. When lifting or stabilizing only use the designated lift
      areas, as shown.  C Gord 3 CPR © 2025 FORD MOTOR COMPANY C Gord
      DEARBORN, MICHIGAN 48121 _ 11/2025  VERSION: Ford Mach-E_2025 11 v01
      Page 9 of 38  F
  [2] p.18-19  type=CompositeElement  imgs=0.0
      Service and handling of Pyrotechnic Components is restricted to
      qualified personnel. The required qualifications vary by region.
      Always observe local laws and legislative directives regarding
      Pyrotechnic Components service and handling. Failure to follow this
      instruction may result in serious personal injury or death.  If
      occupant removal is necess
  [3] p.34-35  type=CompositeElement  imgs=0.0
      Follow the guidel


🔍  [rescue_sheet]  'airbag locations top view'
  [1] p.3-4  type=CompositeElement  imgs=0.0
      FORD Mustang Mach-E 2026  12-Volt Battery Disconnect. (Disables Safety
      Restraint System)  way ™ qa —  Locate and cut the negative battery
      cable to ground at two locations 3 in (7.6 cm) apart from each other.
      Isolate cable to prevent reconnection.  @) Laminated Safety Glass @
      Tempered Safety Glass  A  Refer to the vehicle overview illustration
      on Pa
  [2] p.1-3  type=CompositeElement  imgs=0.0
      C Gord) CPR © 2025 FORD MOTOR COMPANY pends MICHIGAN 48121 VERSION:
      Ford_Mach-E_2025_10_v01  FORD Mustang Mach-E 2026  For more detailed
      information, please refer to the full Mustang Mach-E Emergency
      Response Guide.  1. Identification / recognition  ELECTRIC VEHICLES DO
      NOT HAVE ENGINE NOISE. LACK OF ENGINE NOISE DOES NOT MEAN VEHICLE IS
      OFF: SILEN
  [3] p.4  type=CompositeElement  imgs=0.0
      9. Important additional information  Fo

## Appendix — What changed and why

| Parameter / Code | v1 | v2 (this notebook) | Reason |
|---|---|---|---|
| Image elements | **Silently dropped** — `el.text` is empty for Image type | **Summarized with GPT-4o** via `orig_elements` → `image_base64` | ~50% of ERG content is diagrams (no-cut zones, HV maps, lift points) |
| `include_orig_elements` | Not set | **`True`** | Required flag to make `image_base64` available inside each chunk's metadata |
| `elements_from_base64_gzipped_json` | Not imported | **Imported and used** | Unstructured compresses `orig_elements` as Base64 gzip; must decode before reading |
| Image summary prompt | — | **ERG-specific system prompt** | Generic prompts miss safety callouts; prompt names HV components, no-cut zones etc. |
| `page_span` metadata | Single `page_number` | **`page_span` (e.g. `"12-15"`)** | `by_title` merges multi-page sections; recording all pages aids human review |
| `has_images` / `image_count` | — | **Added to metadata** | Allows Pinecone metadata filtering to restrict queries to image-enriched chunks |
| `combine_under_n_chars` param name | `combine_under_n_chars` | **`combine_text_under_n_chars`** | Correct parameter name per Unstructured `by_title` docs |
